In [16]:
import webcam
import sys
import time
from time import time
import datasetMng
import json
from mqtt_wrapper import MQTTWrapper

# Libraries
import numpy as np
from tqdm import tqdm
from PIL import Image

import torch
from torch.optim import Adam
from torch.nn import CrossEntropyLoss, Conv2d, AvgPool2d, BatchNorm2d, Dropout2d, LeakyReLU, Linear, Module
from torch.utils.data import DataLoader

from torchvision.transforms import transforms
from torchvision.datasets import ImageFolder

In [18]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [20]:
datasetMng.dataTransformer()

In [22]:
# Build Model
class DistractionCNN(Module):
    def __init__(self):
        super(DistractionCNN, self).__init__()
        self.conv = Conv2d(in_channels=3, out_channels=8, kernel_size=(3,3), stride=1, padding=1)
        self.pool = AvgPool2d(kernel_size=(3,3), stride=1)
        self.relu = LeakyReLU()
        self.bn = BatchNorm2d(num_features=8)
        self.drop = Dropout2d(p=0.3)
        self.fc = Linear(in_features=8*126*126, out_features=5)
        
    def forward(self, X):
        output = self.conv(X)
        output = self.pool(output)
        output = self.relu(output)
        output = self.bn(output)
        output = self.drop(output)
#         print(output.shape)
        output = output.view(-1, 8*126*126)
        output = self.fc(output)
        return output

In [24]:
# Define Model and Migrate to GPU
model = DistractionCNN().to(device)
# Optimiser
optimiser = Adam(model.parameters(), lr=1e-4, weight_decay=0.0001)

# Loss Function
loss_fn = CrossEntropyLoss()

In [ ]:
for epoch in range(25):
    start = time()
    train_acc = 0
    test_acc = 0

    # Train
    model.train()
    with tqdm(datasetMng.train_loader, unit="batch") as tepoch:
        for xtrain, ytrain in tepoch:
            optimiser.zero_grad()
            xtrain = xtrain.to(device)
            train_prob = model(xtrain)
            train_prob = train_prob.cpu()
            train_loss = loss_fn(train_prob, ytrain)
            train_loss.backward()
            optimiser.step()
            # END TRAIN
            train_pred = torch.max(train_prob, 1).indices
            train_acc += int(torch.sum(train_pred == ytrain))
            
        train_epoch_accuracy = train_acc / datasetMng.len_train

    # Evaluate
    model.eval()
    with torch.no_grad():
        for xtest, ytest in datasetMng.test_loader:
            xtest = xtest.to(device)
            test_prob = model(xtest)
            test_prob = test_prob.cpu()
            test_loss = loss_fn(test_prob, ytest)
            test_pred = torch.max(test_prob, 1).indices
            test_acc += int(torch.sum(test_pred == ytest))
        
        test_epoch_accuracy = test_acc / datasetMng.len_test
    
    end = time()
    
    diff = end - start
    
    print(f"Epoch: {epoch+1}, Time: {diff}\nTr_loss: {train_loss}, Test_loss: {test_loss}\n,Tr_acc:{train_epoch_accuracy}, Test_acc: {test_epoch_accuracy}")

100%|██████████| 505/505 [01:48<00:00,  4.64batch/s]


Epoch: 1, Time: 121.18741416931152
Tr_loss: 1.997125506401062, Test_loss: 0.7836863994598389
,Tr_acc:0.5357755775577557, Test_acc: 0.6752096924510718


100%|██████████| 505/505 [01:47<00:00,  4.68batch/s]


Epoch: 2, Time: 119.9018566608429
Tr_loss: 0.4041215479373932, Test_loss: 0.5322002172470093
,Tr_acc:0.6692409240924092, Test_acc: 0.7138863000931966


100%|██████████| 505/505 [01:47<00:00,  4.70batch/s]


...

100%|██████████| 505/505 [01:47<00:00,  4.70batch/s]


Epoch: 11, Time: 119.45687246322632
Tr_loss: 0.5366671085357666, Test_loss: 0.7616949081420898
,Tr_acc:0.8356435643564356, Test_acc: 0.8932898415657037


100%|██████████| 505/505 [01:47<00:00,  4.70batch/s]


Epoch: 12, Time: 119.6061224937439
Tr_loss: 0.17472362518310547, Test_loss: 0.06756073981523514
,Tr_acc:0.8355775577557756, Test_acc: 0.8648648648648649


 55%|█████▌    | 278/505 [00:59<00:49,  4.60batch/s]

In [ ]:
def loadUserConfig():
    with open(sys.path[0] + "/userConfig.json") as json_file:
        return json.load(json_file)

client = MQTTWrapper.fromJsonConfig(sys.path[0] + "/mqttConfig.json")
user = loadUserConfig()
webcam.setup()

while True:
    try:
        distractionsMap = webcam.getDistractionType()
        dataToTransmit = { 
            user["user"]: {
                "other_activities": distractionsMap["Other activities"],
                "safe_driving": distractionsMap["Safe driving"],
                "talking_phone": distractionsMap["Talking on the phone"],
                "texting_phone": distractionsMap["Texting on the phone"],
                "turning": distractionsMap["Turning"]
            } 
        }
        client.publish(json.dumps(dataToTransmit))
        print(dataToTransmit)
        time.sleep(1)
    except webcam.NoFaceDetectedException:
        print("No face found!")